In [64]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import decimal as dec
import re

In [65]:
movies = pd.read_csv('datasets/IMDb movies.csv', sep=";", encoding="latin1")
moviesaux = pd.read_csv('datasets/IMDb movies.csv', sep=";", encoding="latin1")
ratings = pd.read_csv('datasets/IMDb ratings.csv', sep=";")
ratingsaux = pd.read_csv('datasets/IMDb ratings.csv', sep=";")

#display(d1.head(10))
#display(d2.head(10))

In [66]:
#Parte da remoção de colunas do Pedro

missing_values_m = moviesaux.isnull().sum()
# print('MOVEIS', missing_values_m)

tamanho = len(movies.index)

moviesaux = moviesaux.drop(['imdb_title_id'], axis=1)

missing_values = movies.isnull().sum()

percentagem = (missing_values / tamanho) * 100

df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentagem': percentagem.round(2)
})

#print(df.to_string())
indice = df[df['Percentagem']> 50 ].index
movies = movies.drop(columns=indice)

#---------------------------------------------------------------

tamanho = len(ratings.index)

ratingsaux = ratingsaux.drop(['imdb_title_id'], axis=1)

missing_values = ratings.isnull().sum()

percentagem = (missing_values / tamanho) * 100

df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentagem': percentagem.round(2)
})

indice = df[df['Percentagem']> 50 ].index
df.drop(indice, inplace=True) 

# print(df.to_string())

dados = ratings[~ratings.isna()]
colunas = [
    "allgenders_0age_votes", "allgenders_18age_votes",
    "allgenders_30age_votes", "allgenders_45age_votes",
    "males_allages_votes", "males_0age_votes", "males_18age_votes",
    "males_30age_votes", "males_45age_votes",
    "females_allages_votes", "females_0age_votes", "females_18age_votes",
    "females_30age_votes", "females_45age_votes",
    "top1000_voters_votes", "us_voters_votes", "non_us_voters_votes"
]

votos_total = dados['total_votes'].sum()
colunas_a_remover = []

for coluna in colunas:
    votos_demografia = dados[coluna].sum()
    proporcao =  votos_demografia / votos_total
    #print('Proporção de votos em percentagem da coluna', coluna, proporcao.round(2)*100)
    if proporcao < .10:
        colunas_a_remover.append(coluna)
    proporcao = 0

ratings = ratings.drop(columns=indice)

In [67]:
dataset = pd.merge(movies, ratings, on="imdb_title_id", how="inner")
pd.set_option('display.max_columns', None)  # Mostra todas as colunas
#display(dataset.head(5))

In [68]:
#Visualização inicial

#print(dataset.shape) #Linhas e colunas do dataset
#print(dataset.describe()) #Min, max, média... de cada coluna
#dataset.describe(include="all()) #O mesmo, mas agora também para as colunas não numéricas
print(dataset.info()) #Tipos de dados presentes + Contagem de valores não nulos
#print(dataset.isnull().sum())) #Contagem de valores nulos
#dataset.drop_duplicates() #Garantia de utilização de apenas valores únicos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85855 entries, 0 to 85854
Data columns (total 60 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   imdb_title_id              85855 non-null  object 
 1   title                      85855 non-null  object 
 2   original_title             85855 non-null  object 
 3   year                       85854 non-null  float64
 4   date_published             85855 non-null  object 
 5   genre                      85855 non-null  object 
 6   duration                   85855 non-null  int64  
 7   country                    85791 non-null  object 
 8   language                   84954 non-null  object 
 9   director                   85768 non-null  object 
 10  writer                     84283 non-null  object 
 11  production_company         81400 non-null  object 
 12  actors                     85786 non-null  object 
 13  description                83740 non-null  obj

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------
#Tratamento de dados
#------------------------------------------------------------------------------------------------------------------------------

#Tratamento de objects
movies = movies.drop(['year'], axis=1) #Temos a coluna date_published que tem mais ou igual informação e não têm missing values


diferentes = dataset[dataset['original_title'] != dataset['title']]
#print(diferentes)
movies = movies.drop(['original_title'], axis=1) #Temos a coluna title que têm a mesma informação mais uniforme

#dataset['date_published '] = pd.to_datetime(dataset['date_published '], format="%d-%m-%Y", errors='coerce')
#Isto resulta para diminuir o tamanho, só que cria missing values


#Tratamento de floats
for coluna in dataset.select_dtypes(include=["float64"]).columns:
    dataset[coluna] = dataset[coluna].astype("float16")
    #if re.search(r"avg|rating|metascore|mean", coluna): 
        #dataset[coluna] = dataset[coluna].astype("float16")
    #else: 
        #dataset[coluna] = dataset[coluna].astype("int64")

#Tratamento de inteiros
for coluna in dataset.select_dtypes(include=["int64"]).columns:
    minimo = dataset[coluna].min()
    maximo = dataset[coluna].max()
    if minimo >= -128 and maximo <= 127:
        dataset[coluna] = dataset[coluna].astype("int8")
    elif minimo >= -32768 and maximo <= 32767:
        dataset[coluna] = dataset[coluna].astype("int16")
    elif minimo >= -2147483648 and maximo <= 2147483647:
        dataset[coluna] = dataset[coluna].astype("int32")

#print(dataset.info()) #Para saber o novo tamanho do dataset e os tipos que mudaram (45,9MB orginalmente)
#display(dataset.head(5))

In [70]:
#Gráficos iniciais

#dataset["top1000_voters_rating"].hist(bins=30, edgecolor="black")
#plt.xlabel("Rating")
#plt.ylabel("Número de Entradas")
#plt.title("Distribuição de votos do top 1000")
#plt.show()

#plt.scatter(dataset["us_voters_rating"], dataset["non_us_voters_rating"], alpha=0.1)
#plt.xlabel("Voters US")
#plt.ylabel("Voters non US")
#plt.title("Dispersão US x non US")
#plt.show()